In [19]:
import pandas as pd

In [20]:
df = pd.read_feather('../dataset/data_andre.feather')
df['date'] = pd.to_datetime(df['date'])

In [21]:
min_date = df['date'].min()
max_date = df['date'].max()
expected_dates = pd.date_range(start=min_date, end=max_date, freq='D')
expected_count = len(expected_dates)

print(f"Global date range: {min_date.date()} to {max_date.date()} ({expected_count} days)")

Global date range: 2021-01-23 to 2023-02-22 (761 days)


In [22]:
date_counts = df.groupby(['item_id', 'store_id'])['date'].nunique().reset_index()
date_counts.rename(columns={'date': 'actual_days'}, inplace=True)
incomplete_products = date_counts[date_counts['actual_days'] < expected_count]

print(f"\nTotal product-store combinations: {len(date_counts)}")
print(f"Total incomplete combinations: {len(incomplete_products)}")

if len(incomplete_products) > 0:
    print("\nSample of incomplete products:")
incomplete_products.head(10)


Total product-store combinations: 1427
Total incomplete combinations: 467

Sample of incomplete products:


,item_id,store_id,actual_days
6,176,6269,760
14,260,6269,759
28,471,6269,739
31,493,6269,759
34,514,6269,751
37,612,6269,760
41,676,6269,760
48,779,6269,734
50,808,6269,760
65,1241,6269,759


In [23]:
unique_items = df[['item_id', 'store_id']].drop_duplicates()
dates_df = pd.DataFrame({'date': expected_dates})
full_grid = unique_items.merge(dates_df, how='cross')
df_fulfilled = full_grid.merge(df, on=['item_id', 'store_id', 'date'], how='left')

In [24]:
df_fulfilled['value'] = df_fulfilled['value'].fillna(0)

In [25]:
promo_cols = [c for c in df_fulfilled.columns if c.startswith('promo_type_') or c.startswith('promo_value_')]
df_fulfilled[promo_cols] = df_fulfilled[promo_cols].fillna(0)

In [26]:
cat_cols = [c for c in df_fulfilled.columns if c in ('cat_label', 'sdep_label', 'dep_label', 'dmn_label')
or c.startswith('cat_label_') or c.startswith('sdep_label_')
or c.startswith('dep_label_') or c.startswith('dmn_label_')]
if cat_cols:
    df_fulfilled[cat_cols] = (
    df_fulfilled.groupby(['item_id', 'store_id'])[cat_cols]
    .transform(lambda s: s.ffill().bfill())
    )

print(f"\nOriginal row count: {len(df)}")
print(f"Fulfilled row count: {len(df_fulfilled)}")


Original row count: 1082371
Fulfilled row count: 1085947


In [27]:
remaining_nans = df_fulfilled.isnull().sum()
remaining_nans = remaining_nans[remaining_nans > 0]
if len(remaining_nans):
    print(f"\nRemaining NaNs after imputation:\n{remaining_nans}")
else:
    print("\nNo remaining NaNs.")

df_fulfilled.to_feather('../dataset/data_andre_fulfilled.feather')


No remaining NaNs.
